# Agent 및 Task 설계와 프롬프트 작성법

## 실습 목표
- **CrewAI 프레임워크**를 활용하여 **다중 에이전트 시스템**을 구성하고 여행 일정 생성 예제를 통해 실습합니다.
- **Agent**(에이전트)의 역할(Role), 목표(Goal), 배경(Backstory)을 정의하고 **Task**(태스크)의 설명(프롬프트)과 기대 출력(Expected Output)을 작성하는 방법을 익힙니다.
- 여러 에이전트가 **협업**하여 작업을 수행하는 과정(예: 정보 조사 에이전트 + 일정 기획 에이전트)을 단계별로 경험합니다.
- 프롬프트에 추가 정보를 요구하거나 제약조건을 넣어 **출력 결과의 품질을 개선**하는 방법을 학습합니다.

## 사전 요구사항
- **Python 3.x**가 설치되어 있어야 합니다 (권장: Python 3.10 이상).
- **CrewAI 라이브러리**가 설치되어 있어야 합니다 (터미널에서 `pip install crewai` 및 추가 도구를 위해 `pip install "crewai[tools]"` 실행).
- 인터넷 연결: 실습 코드 중 **웹 검색 도구**를 사용하는 부분이 있으므로 인터넷 접속이 가능해야 합니다.
- OpenAI API 키: OpenAI의 GPT 모델을 사용하므로, **`.env` 파일**에 `OPENAI_API_KEY`를 설정해야 합니다 (없으면 코드 실행 시 오류 발생).
- 코드 편집기/실행 환경: Visual Studio Code (VS Code)와 같은 IDE 또는 터미널에서 Python 파일을 실행할 수 있는 환경.

> **NOTE:** `.env` 파일은 프로젝트 루트에 위치시키고, 내용에 `OPENAI_API_KEY=<YOUR_API_KEY>` 형식으로 OpenAI API 키를 적어두세요. CrewAI의 LLM으로 OpenAI의 GPT-4 모델(`gpt-4o-mini`)을 사용하므로 해당 키가 필요합니다.

---

## 03-1.travel_planner.py : 단일 에이전트로 여행 일정 생성

**시나리오:** CrewAI 프레임워크로 **부산 3일 여행 일정**을 자동 생성하는 단일 에이전트 시스템을 구현합니다. 한 명의 "여행 기획자" 에이전트가 주어진 요청에 따라 일정을 계획합니다.

### 코드 설명

먼저 필요한 라이브러리를 불러오고 환경변수를 설정합니다. `.env`에서 OpenAI API 키를 로드하고, **LLM**(Large Language Model)을 초기화합니다. 이 예제에서는 OpenAI의 GPT-4 모델의 미니 버전을 사용하며, 온도(temperature)는 0.7로 설정했습니다 (출력의 창의성 정도 조절).

```python
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM

# .env 파일 로드하여 OPENAI_API_KEY 등 환경변수 설정
load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise EnvironmentError("OPENAI_API_KEY not found. Please set it in a .env file.")

# LLM 초기화 (CrewAI LLM 래퍼 사용)
llm = LLM(model="openai/gpt-4.1-mini", temperature=0.7)
```

이어서 **여행 기획자 에이전트**를 정의합니다. `Agent` 클래스를 사용하여 에이전트를 생성하며, 다음과 같은 속성을 부여합니다:

- `role`: 에이전트의 역할 명칭 (예: `"여행 기획자"`).
- `goal`: 에이전트의 목표 또는 임무 설명. 여기서는 *"사용자의 요청에 따라 여행 일정을 계획하고 제안합니다."*와 같이 정의했습니다.
- `backstory`: 에이전트의 배경 설정으로, 실제 성능에는 직접 영향은 없지만 에이전트의 행동 스타일에 간접 영향을 줄 수 있습니다. 예에서는 *"여행사에서 10년 경력의 전문 여행 플래너"*로 설정했습니다.
- `llm`: 앞서 초기화한 LLM 객체를 지정합니다 (이 에이전트가 사용할 언어 모델).
- `verbose`: True로 설정하면 동작 과정을 상세히 콘솔에 출력합니다 (디버깅이나 학습용으로 유용).

```python
# 여행 기획자 에이전트 정의
travel_agent = Agent(
    role="여행 기획자",
    goal="사용자의 요청에 따라 여행 일정을 계획하고 제안합니다.",
    backstory="여행사에서 10년 경력의 전문 여행 플래너로, 다양한 국내 여행 코스를 알고 있습니다.",
    llm=llm,
    verbose=True
)
```

이제 **여행 일정 작성 작업(Task)** 을 정의합니다. `Task` 클래스에는 주어진 작업에 대한 **프롬프트(description)** 와 해당 작업을 수행할 에이전트, 그리고 예상 출력 형식을 기술합니다:

- `description`: 에이전트에게 주어지는 지시사항 또는 프롬프트입니다. 문자열로 작성하며, 여러 줄에 걸쳐 상세 요구사항을 기술할 수 있습니다.  
  **➡ 예시 프롬프트:** 부산에서 3일간 여행 일정을 계획하도록 요청하면서, *"1일차, 2일차, 3일차로 나누고 각 일자마다 아침/점심/저녁에 할 활동을 상세히 제안"*하도록 요구했습니다. 또한 *"부산의 주요 관광지와 현지 맛집 추천 포함, 교통 수단 정보나 팁이 있으면 제공"* 등 세부 요구사항도 포함되어 있습니다.
- `agent`: 이 Task를 수행할 책임이 있는 에이전트를 지정합니다 (`travel_agent`를 연결).
- `expected_output`: 예상되는 출력 형태나 내용에 대한 요약을 적습니다. (예: *"Day 1, Day 2, Day 3으로 구분된 상세 일정 제안"*). 엄격한 검증 용도는 아니지만, 어떤 결과를 기대하는지 표현하여 나중에 결과를 확인하거나 디버깅할 때 도움을 줍니다.

```python
# 부산 3일 여행 일정 작성 Task 정의
itinerary_task = Task(
    description=(
        "부산에서 3일간 여행 일정을 계획해 주세요.\n"
        "1일차, 2일차, 3일차로 나누고, 각 일자마다 아침/점심/저녁에 할 활동을 상세히 제안하세요.\n"
        "여행 일정에는 부산의 주요 관광지와 현지 맛집 추천을 포함하고, 교통 수단 정보나 팁이 있으면 함께 제공하세요."
    ),
    agent=travel_agent,
    expected_output="Day 1, Day 2, Day 3으로 구분된 상세 일정 제안"
)
```

마지막으로 **Crew**를 구성하고 실행합니다. Crew는 에이전트와 태스크를 묶어 전체 프로세스를 관리하는 객체입니다. 여기서는 에이전트 리스트에 `travel_agent` 하나, 태스크 리스트에 `itinerary_task` 하나만 넣었습니다. `process=Process.sequential`로 설정하여 Task들을 순차적으로 실행하도록 합니다 (지금은 태스크가 하나라 자동으로 순차 실행됩니다). 

`crew_single.kickoff()`를 호출하면 Crew에 속한 태스크들이 정의된 순서에 따라 실행됩니다. 실행 결과는 `result_single` 변수에 저장되며, 이어서 콘솔에 출력합니다.

```python
# Crew 생성 및 실행 (순차 실행 - Task가 하나뿐이므로 순차 처리)
crew_single = Crew(
    agents=[travel_agent],
    tasks=[itinerary_task],
    process=Process.sequential,
    verbose=True
)
print("=== [단일 에이전트] 부산 3일 일정 생성 시작 ===")
result_single = crew_single.kickoff()
print("=== [단일 에이전트] 생성된 부산 3일 일정 ===")
print(result_single)
```

### 실행 방법

1. 터미널에서 해당 파일이 있는 디렉토리로 이동합니다.
2. 다음 명령을 실행하여 코드를 실행합니다:
   ```bash
   python 03-1.travel_planner.py
   ```
3. 실행하면 OpenAI API를 통해 에이전트가 프롬프트에 답을 생성하고, 최종적으로 **부산 3일 여행 일정** 텍스트가 출력됩니다. (API 키 미설정 등 문제가 있다면 오류가 발생하니, 사전 요구사항을 확인하세요.)

### 예상 실행 결과

콘솔에 에이전트의 **프롬프트 응답 결과**가 출력됩니다. 예를 들어, 다음과 같이 **Day 1**, **Day 2**, **Day 3**로 구분된 여행 일정이 생성될 것입니다:

```
=== [단일 에이전트] 생성된 부산 3일 일정 ===
Day 1:
- **아침:** 해운대 해수욕장에서 일출 감상. 주변 카페에서 간단한 조식.
- **점심:** 해운대 시장 탐방 후 돼지국밥 맛집에서 점심 식사.
- **저녁:** 광안리 해변 산책 및 광안대교 야경 감상. 해변 근처 횟집에서 저녁 식사.

Day 2:
- **아침:** 감천문화마을 방문하여 벽화마을 골목 산책.
- **점심:** 자갈치 시장에서 신선한 해산물로 식사.
- **저녁:** 남포동 BIFF 거리 구경 후 부산역 인근 밀면 맛집에서 저녁.

Day 3:
- **아침:** 태종대 공원에서 등대 산책 및 경치 감상.
- **점심:** 송도 해상케이블카 이용 후 송도 해수욕장 근처 음식점에서 식사.
- **저녁:** 부산 대표 음식인 씨앗호떡 등 길거리 간식 맛보고 귀경 준비.
```

*예시 출력*은 이해를 돕기 위한 것으로, 실행할 때마다 세부 내용은 달라질 수 있지만 **각 날의 오전/점심/저녁으로 구성된 일정**과 **주요 관광지, 음식점, 활동** 등이 포함된다는 점은 공통적입니다. 또한 에이전트가 자체 지식에 기반해 일정을 생성하므로, 최신 정보나 실제 이동 시간 등은 반영되지 않을 수 있습니다 (이 부분을 개선하기 위해 다음 실습에서 정보 검색을 도입할 것입니다).

---

## 03-2.travel_planner.py : 두 에이전트 협업으로 일정 생성

**시나리오:** 이번에는 **두 명의 에이전트**가 협업하여 부산 3일 일정을 작성합니다. 하나의 에이전트는 정보를 **검색**하고, 다른 에이전트는 그 정보를 활용해 **일정 작성**을 합니다. 이를 통해 **역할 분담**과 **태스크 간 컨텍스트 공유**를 배우겠습니다.

### 코드 설명

이전 실습과 마찬가지로 환경설정 (env 로드, LLM 초기화)은 동일하게 이뤄집니다. 핵심 차이는 두 가지입니다:
1. **웹 검색 도구(Tool)**의 도입과 정보 조사 에이전트에 해당 도구 장착
2. 두 개의 에이전트와 두 개의 태스크를 **순차적으로 실행**하고 앞 태스크의 결과를 다음 태스크에 **전달**하는 구조

#### 1. 커스텀 웹 검색 도구 준비

CrewAI에서는 에이전트에 **도구(Tool)**를 장착하여 외부 기능을 사용할 수 있습니다. 여기서는 CrewAI 공식 도구 패키지의 `SerperDevTool`을 만들고 정보 조사자가 사용하도록 합니다. 이 도구는 `SERPER_API_KEY` 환경변수를 사용해 웹 검색을 수행합니다. 이렇게 함으로써 에이전트는 프롬프트를 해결하기 위해 인터넷 검색을 할 수 있게 됩니다.

```python
from crewai_tools import SerperDevTool

# 웹 검색 도구 인스턴스 생성
search_tool = SerperDevTool()
```

#### 2. 두 에이전트 정의 (정보 조사자 & 일정 기획자)

이제 **정보 조사자 에이전트**와 **여행 일정 기획자 에이전트** 두 명을 정의합니다. 

- `research_agent` (정보 조사자): 최신 여행 정보를 찾아 제공하는 역할입니다. 앞서 만든 `search_tool`을 `tools` 매개변수에 넣어, 이 에이전트가 프롬프트를 처리하는 중에 웹 검색을 활용할 수 있게 합니다. 목표는 *"여행에 필요한 최신 정보를 조사하여 제공"*하는 것이며, 배경은 *"온라인 정보 검색에 능통한 여행 정보 전문가"*로 설정했습니다.

- `planner_agent` (일정 기획자): 제공된 정보를 활용해 완성도 높은 일정을 작성하는 역할입니다. 이 에이전트는 주어진 컨텍스트(앞서 조사된 정보)를 바탕으로 일정을 계획합니다. 목표는 *"제공된 정보를 활용해 완성도 높은 여행 일정을 작성"*하는 것이며, 배경은 *"국내 여행 일정을 여러 차례 기획한 경험이 풍부한 전문가"*로 설정했습니다. 이 에이전트는 별도의 도구 없이 LLM만 사용합니다.

```python
# 정보 조사 에이전트 정의
research_agent = Agent(
    role="정보 조사자",
    goal="여행에 필요한 최신 정보를 조사하여 제공합니다.",
    backstory="온라인 정보 검색에 능통한 여행 정보 전문가입니다.",
    llm=llm,
    tools=[search_tool],    # 웹 검색 도구 장착
    verbose=True
)

# 일정 작성 에이전트 정의
planner_agent = Agent(
    role="여행 일정 기획자",
    goal="제공된 정보를 활용해 완성도 높은 여행 일정을 작성합니다.",
    backstory="국내 여행 일정을 여러 차례 기획한 경험이 풍부한 전문가입니다.",
    llm=llm,
    verbose=True
)
```

#### 3. 두 개의 Task 정의 및 연결

이제 두 개의 Task를 만듭니다. **첫 번째 Task**는 정보 조사자가 수행할 내용이고, **두 번째 Task**는 일정 기획자가 수행할 내용입니다. 

- `research_task`: 부산 여행에 필요한 핵심 정보를 조사하는 작업입니다. `description`에 예시로 *"부산의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료 기반으로 정리"*하도록 지시하고 있습니다. 이 Task는 `agent=research_agent`로 설정되어 있어 **정보 조사자**에게 할당됩니다. `expected_output`에는 *"부산 여행에 대한 요약 정보 목록"*이라고 적어, 이 작업의 결과로 부산 여행과 관련된 요약 정보(장소 리스트, 팁 등)가 나올 것으로 기대함을 명시했습니다.

- `planning_task`: 위에서 조사된 정보를 참고하여 실제 3일 일정 표를 짜는 작업입니다. `description`에서는 *"위의 조사 결과를 참고하여 부산 3일 여행 일정을 작성"*하고, 각 날짜별 **오전/오후/저녁 계획**을 세우며, **조사된 관광지와 맛집 정보를 일정에 반영**하도록 요청합니다. 또한 *"일정에는 방문지에 대한 간단한 설명이나 여행 팁도 포함"*하도록 요구하여, 단순 나열이 아니라 부가 설명도 달도록 했습니다. 이 Task는 `agent=planner_agent`로 설정되어 **일정 기획자**가 담당합니다. **중요한 부분**은 `context=[research_task]`로 설정한 것입니다. 이것은 **이 Task를 실행할 때 `research_task`의 결과를 컨텍스트로 전달**한다는 뜻입니다. 즉, 일정 기획자 에이전트는 프롬프트를 생성할 때 이전에 조사된 정보를 참고할 수 있게 됩니다. `expected_output`에는 *"조사된 정보를 반영한 3일간의 여행 일정"*이라고 기대 결과를 작성했습니다.

```python
# 정보 조사 Task 정의
research_task = Task(
    description=(
        "부산 여행을 위해 알아야 할 핵심 정보를 조사하세요.\n"
        "부산의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요."
    ),
    agent=research_agent,
    expected_output="부산 여행에 대한 요약 정보 목록"
)

# 일정 작성 Task 정의 (이전 Task 결과를 context로 활용)
planning_task = Task(
    description=(
        "위의 조사 결과를 참고하여 부산에서 3일 동안 머무는 여행 일정을 작성해 주세요.\n"
        "각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요.\n"
        "일정에는 방문지에 대한 간단한 설명이나 여행 팁도 포함해 주세요."
    ),
    agent=planner_agent,
    context=[research_task],  # 이전 조사 결과를 컨텍스트로 전달
    expected_output="조사된 정보를 반영한 3일간의 여행 일정"
)
```

여기서 `context=[research_task]` 설정에 주목하세요. CrewAI에서는 이렇게 앞선 Task의 결과를 다음 Task에 넘겨줄 수 있습니다. 내부적으로는 **research_task의 출력이 planning_task에 입력**으로 제공되어, 일정 기획자 LLM 프롬프트에 자동으로 포함됩니다. 따라서 두 번째 에이전트는 첫 번째 에이전트가 찾아준 최신 정보를 활용해 일정을 세울 수 있습니다.

#### 4. Crew 실행 설정 및 실행

두 에이전트와 두 태스크를 준비했으니, Crew를 생성합니다. 에이전트 리스트에는 `[research_agent, planner_agent]`를, 태스크 리스트에는 `[research_task, planning_task]`를 지정합니다. `process=Process.sequential`로 하여 **순차 처리**를 명시합니다 (첫 번째 태스크 완료 후 두 번째 태스크 실행).

```python
# 두 에이전트를 Crew로 묶어 순차 실행
crew_multi = Crew(
    agents=[research_agent, planner_agent],
    tasks=[research_task, planning_task],
    process=Process.sequential,
    verbose=True
)

print("\n=== [협업 에이전트] 부산 3일 일정 생성 시작 ===")
result_multi = crew_multi.kickoff()
print("=== [협업 에이전트] 생성된 부산 3일 일정 ===")
print(result_multi)
```

`crew_multi.kickoff()`를 호출하면 먼저 **정보 조사자 에이전트**가 `research_task`를 수행합니다. 이 과정에서 에이전트는 내부적으로 필요한 정보를 검색하기 위해 우리가 장착한 `SerperDevTool`을 사용할 것입니다. (예: 부산 관광지, 맛집 등을 검색하고 요약 정리). 그 결과가 나오면, **일정 기획자 에이전트**가 `planning_task`를 수행하며, 첫 번째 결과를 컨텍스트로 활용해 최종 일정을 작성합니다. `result_multi`에는 최종 생성된 일정이 반환되며 이를 출력합니다.

### 실행 방법

1. 터미널에서 해당 파일 디렉토리로 이동합니다.
2. 다음 명령어를 실행하세요:
   ```bash
   python 03-2.travel_planner.py
   ```
3. 실행 과정에서 **정보 조사자** 에이전트가 웹 검색을 수행하므로, 콘솔에 검색 키워드나 중간 결과 등이 `verbose=True`로 인해 출력될 수 있습니다. 이어서 **일정 기획자**의 결과가 나오며, 최종적으로 부산 3일 여행 일정이 완성됩니다.

### 예상 실행 결과

이번에는 **두 단계의 출력**이 이뤄집니다. 첫 번째 에이전트의 **조사 결과**와 두 번째 에이전트의 **최종 일정**인데, 코드에서는 최종 일정(`planning_task`의 결과)만 `print`하고 있습니다. 그러나 `verbose=True` 설정으로 인해 실행 중간에 검색 과정 로그나 1차 조사 내용이 나타날 수 있습니다.

최종 출력되는 일정은 03-1 버전과 유사한 **3일치 여행 일정**이지만, 내용 면에서 더 **최신 정보**나 **구체적인 장소 이름**이 반영되어 있을 것입니다. 예를 들어, 정보 조사 결과에 따라 실제 부산의 최신 인기 카페나 새로운 관광지가 언급될 수 있고, 일정에 그 내용이 반영됩니다. 또한 각 장소에 대한 간단한 설명이나 팁도 포함되어 한층 풍부한 일정표가 될 것입니다.

> **예상 예시**: Day 1에 해운대 해수욕장 방문이 포함되면서 "*해운대 해수욕장은 부산의 대표 해변으로, 아침 산책을 하기에 좋습니다.*"와 같은 설명이 붙고, 맛집으로 최근 현지인들에게 인기있는 식당 이름이 언급될 수 있습니다. 이러한 디테일은 첫 번째 에이전트의 조사에 달려 있으며, 두 번째 에이전트는 그 정보를 활용하게 됩니다.

**비교:** 03-1의 결과와 03-2의 결과를 비교해 보면, 03-2 (협업 에이전트 버전)가 **보다 실제 정보에 기반한 일정**을 제시할 가능성이 높습니다. 즉, 단일 에이전트가 자체 지식으로 만든 일정보다, 하나의 에이전트가 정보를 찾아주고 다른 하나가 일정을 짜는 협업 방식이 **더 신뢰도 높은 세부사항**을 제공하게 됩니다.

---

## 03-3.travel_planner.py : 개선된 프롬프트로 일정 품질 향상

**시나리오:** 두 에이전트 협업 구조는 03-2와 동일하지만, **프롬프트**를 개선하여 **더 현실적이고 상세한 일정**을 만들어 봅니다. 특히 예산과 교통 수단 등의 제약 조건을 프롬프트에 추가로 명시하여, 에이전트가 이를 고려한 계획을 세우도록 유도합니다.

### 코드 설명

이 코드의 구조는 03-2와 거의 같으며, **에이전트 정의와 1차 정보 조사 Task**는 동일합니다. 달라진 점은 **두 번째 Task의 description (프롬프트)** 부분입니다. 즉, **일정 작성 Task**에 추가 지침을 줌으로써 결과의 질을 높이려는 것입니다.

**개선된 일정 작성 Task 정의:** `improved_planning_task`의 `description`에는 이전과 동일한 내용에 더하여 **여행 예산과 대중교통 이용에 대한 조건**이 포함됩니다. 구체적으로 추가/변경된 요구사항은 다음과 같습니다:

- **예산 제한:** *"가능하면 예산은 하루 10만원 내외로 맞추고"* – 하루 예산을 약 10만원 정도로 고려하라는 지시입니다. 이를 통해 에이전트는 지나치게 비싼 활동을 지양하고, 비용 정보를 염두에 둘 것입니다.
- **교통 수단:** *"이동은 모두 대중교통을 이용"* – 택시나 렌터카 대신 버스, 지하철 등 대중교통으로 다니도록 계획하게 합니다.
- **교통 경로 제안:** *"버스정류장 및 지하철역을 포함한 대중교통 경로를 제안해 주세요."* – 이동 시 어떤 버스나 지하철을 탈지, 어디서 승차/하차하는지 등의 구체적인 경로를 제시하도록 합니다.
- **이동 소요 시간:** *"버스 및 지하철을 이용할 때의 소요 시간도 포함해 주세요."* – 각 이동 구간에 걸리는 시간까지 언급해주면 일정 계획에 현실감을 더할 수 있습니다.
- **결과 언어 명시:** *"결과는 한국어로 작성해 주세요."* – 혹시 모를 영문 출력 등을 방지하고, 결과를 반드시 한국어로 작성하도록 명시했습니다 (현재 프롬프트 자체가 한국어이므로 출력도 한국어일 가능성이 높지만, 요구사항을 분명히 하는 차원).

아래는 개선된 두 번째 Task 정의 부분입니다:

```python
# 개선된 일정 작성 Task 정의 (예산 및 교통 고려 추가)
improved_planning_task = Task(
    description=(
        "위의 조사 결과를 참고하여 부산에서 3일 동안 머무는 여행 일정을 작성해 주세요.\n"
        "각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요.\n"
        "가능하면 **예산은 하루 10만원 내외로 맞추고, 이동은 모두 대중교통**을 이용하는 것으로 고려하세요.\n"
        "버스정류장 및 지하철역을 포함한 대중교통 경로를 제안해 주세요.\n"
        "버스 및 지하철을 이용할 때의 소요 시간도 포함해 주세요.\n"
        "일정에는 방문지에 대한 간단한 설명이나 여행 팁도 포함해 주세요. 결과는 한국어로 작성해 주세요.\n"
    ),
    agent=planner_agent,
    context=[research_task],
    expected_output="예산과 교통을 고려한 3일간의 여행 일정"
)
```

위 코드에서 볼 수 있듯이, **粗체로 강조된 부분**(예산 10만원, 대중교통 이용)이 새로 추가되었습니다. 이러한 세부 지침으로 인해 에이전트는 일정을 세울 때 비용과 교통을 신경 쓰게 됩니다. (프롬프트 문자열 내에 `**...**`로 감싼 부분은 마크다운 문법으로 **강조**를 의미하며, 모델에 해당 부분을 강조하여 인식시키기 위한 기법입니다.)

나머지 구성은 03-2와 동일합니다. 동일한 `research_task`를 사용하고, 에이전트들도 동일합니다. Crew 설정 시 이번에는 `tasks=[research_task, improved_planning_task]`로 두 번째 태스크만 바뀐 채 추가됩니다. 실행 순서는 여전히 sequential입니다.

```python
crew_multi_improved = Crew(
    agents=[research_agent, planner_agent],
    tasks=[research_task, improved_planning_task],
    process=Process.sequential,
    verbose=True
)
print("\n=== [협업 에이전트] 개선된 프롬프트로 일정 생성 ===")
result_multi_improved = crew_multi_improved.kickoff()
print("=== [협업 에이전트] 개선된 일정 결과 ===")
print(result_multi_improved)
```

### 실행 방법

이전과 마찬가지로 터미널에서 다음 명령을 실행합니다:
```bash
python 03-3.travel_planner.py
```
실행 과정 역시 03-2와 유사하게 진행되지만, 두 번째 에이전트의 프롬프트가 강화되었으므로 출력까지 약간 더 시간이 걸릴 수 있습니다 (LLM이 더 많은 내용을 생성할 수 있으므로). 

### 예상 실행 결과

출력되는 **부산 3일 여행 일정**은 이전보다 **더 현실적이고 상세한 정보**를 담고 있을 것입니다. 특히, 각 이동 단계에 **대중교통 경로와 시간**, 그리고 **비용 측면**이 언급될 것으로 기대됩니다. 예를 들어 일정의 일부는 다음과 같을 수 있습니다:

```
Day 2:
- **오전:** 감천문화마을 투어 (지하철 1호선 토성역 하차, 7번 출구에서 마을 버스 2번 이용, 약 15분 소요). 알록달록한 벽화를 감상하며 산책.
- **오후:** 자갈치 시장 방문 및 점심 (토성역에서 1호선 이용, 자갈치역 하차, 도보 5분). 신선한 회 시식 예산 약 2만원. 
- **저녁:** 광안리 해변에서 야경 감상 (자갈치에서 버스 41번 탑승 후 광안리해변 정류장 하차, 약 30분 소요). 해변 근처 식당에서 돼지국밥으로 저녁 (~8천원).
```

위는 *예상 예시*로, 실제 결과와 다를 수 있지만 **형식**은 유사할 것입니다. 개선된 프롬프트 덕분에 일정마다:
- 교통: *(어느 역에서 몇 번 버스를 타고, 몇 분 걸리는지)* 같은 내용이 포함되고,
- 예산: *(식사에 얼마 정도 비용이 드는지)* 등의 언급이 추가됩니다.

또한 모든 출력은 한국어로 명시적으로 요구했으므로 한글로 제공될 것이 확실해졌습니다. 전체적으로 사용자는 **보다 구체적이고 실행 가능한 여행 일정**을 얻게 됩니다.

---

## 정리 및 팁

이상으로, **CrewAI를 활용한 Agent 및 Task 설계, 그리고 프롬프트 작성 실습**을 완료했습니다. 이번 실습에서 배운 주요 사항을 정리하면:

- **Agent 정의:** 역할, 목표, 배경을 명확히 설정함으로써 에이전트의 행동 방향을 결정할 수 있습니다. 필요한 경우 도구를 부여하여 정보 검색이나 계산 등의 능력을 확장할 수 있습니다.
- **Task 정의:** 프롬프트(설명)를 구체적으로 작성하고, 어떤 출력물을 기대하는지 `expected_output`으로 표현해 두면 좋습니다. 여러 Task가 있을 때 이전 Task를 `context`로 넘겨줄 수 있어, **태스크 간에 정보 공유**가 가능합니다.
- **Crew 활용:** Crew 객체로 에이전트들과 태스크들을 엮어 일괄 실행할 수 있습니다. `Process.sequential`을 사용하면 지정된 순서대로 태스크를 실행하고, 각 단계의 결과를 다음 단계에 전달할 수 있습니다.
- **프롬프트 개선:** 출력 결과를 향상시키기 위해서는 프롬프트에 **명확한 요구사항과 제약**을 추가하는 것이 효과적입니다. 요구사항이 구체적일수록 에이전트(LLM)는 더 관련성 높은 세부정보를 포함한 답변을 생성합니다. 다만 과도하게 제한하면 창의성이 떨어질 수 있으므로 균형을 잡는 것이 중요합니다.

이 실습을 통해 간단한 여행 일정 플래너를 만들어보았지만, 동일한 개념을 활용하면 다양한 도메인에 에이전트와 태스크를 설계하여 복잡한 문제를 해결할 수 있습니다. **CrewAI**를 활용한 에이전트 기반 시스템 설계에 익숙해졌기를 바랍니다. 즐거운 여행 플래닝 실습이었길 바랍니다!


In [1]:
from config import CONTENT_DIR, GOOGLE_AI_API_KEY, OPENAI_API_KEY, SERPER_API_KEY
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

gemma = 'gemini/gemma-4-26b-a4b-it'
gpt_4o_mini = 'gpt-4o-mini'
gemini_flash_lite = 'gemini/gemini-3.1-flash-lite'
llm = LLM(
    model=gemini_flash_lite,
    temperature=0.4,
    api_key=GOOGLE_AI_API_KEY,
    max_tokens=1000,
    top_p=0.9
)


In [ ]:
travel_agent = Agent(
    role ='여행 기획자',
    goal = '사용자의 요청에 따라 통영 여행 일정을 계획하고 제안합니다.',
    backstory='여행사에서 10년 경력의 전문 여행 플래너로, 다양한 국내 여행 코스를 알고 있습니다.',
    llm=llm,
    verbose=True
)
itinerary_task = Task(
    description=(
        "통영에서 3일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다 아침/점심/저녁에 할 활동을 상세히 제안하세요."
        "여행 일정에는 통영의 주요 관광지와 현지 맛집 추천을 포함하고, 교통 수단 정보나 팁이 있으면 함께 제공하세요."
    ),
    agent=travel_agent,
    expected_output="3일을 Day1, Day2, Day3으로 구분된 상세 일정 제안"
)

crew_single = Crew(
    agents=[travel_agent],
    tasks=[itinerary_task],
    process=Process.sequential,
    verbose=False
)
result_single = await crew_single.kickoff_async()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 기획자                                                                                             │
│                                                                                                                 │
│  Task: 통영에서 3일간 여행 일정을 계획해주세요. 1일 이상이면 1일차, 2일차, 3일차로 나누고, 각 일자마다          │
│  아침/점심/저녁에 할 활동을 상세히 제안하세요.여행 일정에는 통영의 주요 관광지와 현지 맛집 추천을 포함하고,     │
│  교통 수단 정보나 팁이 있으면 함께 제공하세요.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 기획자                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  안녕하세요! 10년 차 여행 플래너입니다. '동양의 나폴리'라 불리는 아름다운 항구 도시, 통영에서의 2박 3일 여행을  │
│  완벽하게 즐기실 수 있도록 알찬 일정을 구성해 드립니다.                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **[통영 2박 3일 낭만 여행 코스]**                                                                          │
│                                                                                                                 │
│  #### **Day 1: 통영의 역사와 예술을 걷다**                                                                      │
│  *   **아침: 통영 도착 및 '서피랑' 산책**                                                                       │
│      *   활동: 서피랑 공원과 99계단을 오르며 통영 강구안 항구의 전경을 한눈에 담아보세요.                       │
│      *   맛집: **'통영식 시락국'** (서호시장 인근). 멸치 육수로 끓여낸 통영식 시래기국은 아침 식사로            │
│  최고입니다.                                                                                                    │
│  *   **점심: 통영의 맛, 충무김밥**                                                                              │
│      *   활동: 강구안 문화마당 주변을 산책하며 통영의 랜드마크인 거북선 관람.                                   │
│      *   맛집: **'뚱보할매김밥'** 등 강구안 인근 충무김밥 거리에서 석박지와 오징어무침을 곁들인 원조            │
│  충무김밥을 맛보세요.                                                                                           │
│  *   **저녁: 동피랑 벽화마을과 야경**                                                                           │
│      *   활동: 아기자기한 벽화가 가득한 동피랑 마을을 둘러보고, 정상 '동포루'에서 노을을 감상하세요.            │
│      *   맛집: **'통영 다찌집'** (강구안 인근). 통영만의 독특한 술상 문화인 '다찌'를 경험해보세요. 제철         │
│  해산물이 끊임없이 나옵니다.                                                                                    │
│                                                                                                                 │
│  #### **Day 2: 바다 위 통영의 절경을 찾아서**                                                                   │
│  *   **아침: 미륵산 케이블카와 루지 체험**                                                                      │
│      *   활동: 통영 케이블카를 타고 미륵산 정상에 올라 한려수도의 비경을 감상한 뒤, 내려와서 스카이라인 루지를  │
│  타며 짜릿한 액티비티를 즐기세요.                                                                               │
│  *   **점심: 통영 굴 요리**                                                                                     │
│      *   맛집: **'대풍관'** 등 미륵도 인근의 굴 요리 전문점. 굴 코스 요리(굴전, 굴무침, 굴밥 등)로 통영 바다의  │
│  맛을 제대로 느껴보세요.                                                                                        │
│  *   **저녁: 달아공원 일몰과 해안 드라이브**                                                                    │
│      *   활동: 통영 최고의 일몰 명소인 '달아공원'에서 바다로 지는 해를 감상하세요.                              │
│      *   맛집: **'해물뚝배기'** 전문점. 신선한 해산물이 가득 들어간 뚝배기로 든든한 저녁을 마무리하세요.        │
│                                                                                                                 │
│  #### **Day 3: 섬 여행의 여유와 기념품 쇼핑**                                                                   │
│  *   **아침: 욕지도 또는 연대도/만지도 섬 여행**                                                                │
│      *   활동: 통영 여객선 터미널에서 배를 타고 인근 섬으로 떠나보세요. 특히 연대도-만지도 출렁다리는 걷기      │
│  좋고 경치가

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [2]:
search_tool = SerperDevTool()

# 정보 조사 에이전트 정의
research_agent = Agent(
    role='정보 조사자',
    goal='{place} 여행에 필요한 최신 정보를 조사하여 제공합니다.',
    backstory='온라인 정보 검색에 능통한 여행 정보 전문가입니다.',
    llm=llm,
    tools=[search_tool],        # 웹 검색 도구 장착
    verbose=True
)

# 일정 작성 에이전트 정의
planner_agent = Agent(
    role ='여행 일정 기획자',
    goal = '제공된 정보를 활용해서 완성도 높은 {place} 여행 일정을 작성합니다.',
    backstory='여행사에서 10년 경력의 전문 여행 플래너로, 다양한 국내 여행 일정을 여러 차례 기획한 경험이 있습니다.',
    llm=llm,
    verbose=True
)

research_task = Task(
    description=(
        "{place} 여행을 위해 알아야 할 핵심 정보를 조사하세요. \n"      # \n : 줄 바꿈
        "{place}의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요."
    ),
    agent=research_agent,
    expected_output="한국어로 작성된 {place} 여행에 대한 요약 정보 목록"
)

planning_task = Task(
    description=(
        "위의 조사 결과를 참고하여 {place}에서 {days}일 동안 머무는 여행 일정을 작성해 주세요. \n"     
        "각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요.\n"
        "일정에는 방문지에 대한 간단한 설명이나 여행 팁도 포함해 주세요."
    ),
    agent=planner_agent,
    context=[research_task],        # 이전 조사 결과를 컨텍스트로 전달
    expected_output="한국어로 작성된 정보를 반영한 {days} 일간의 여행 일정"
)
# 두 에이전트를 Crew로 묶어 순차 진행
crew_multi = Crew(
    agents=[research_agent, planner_agent],
    tasks=[research_task, planning_task],
    process=Process.sequential,
    verbose=True
)
place = "제주"
days = 3

print(f"=== [협업 에이전트] {place} {days}일 일정 생성 시작 ===")
result_multi = await crew_multi.kickoff_async(inputs = {'place':place, 'days':days})
print(f"=== [협업 에이전트] 생성된 {place} {days}일 일정 ===")
print(result_multi)

=== [협업 에이전트] 제주 3일 일정 생성 시작 ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97c0f2c3-505c-44c8-8c46-45692c5e50eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 제주 여행을 위해 알아야 할 핵심 정보를 조사하세요.                                                       │
│  제주의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요.   │
│  ID: ecb965a0-83cb-4c16-8be8-f593e3d252f9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 조사자                                                                                             │
│                                                                                                                 │
│  Task: 제주 여행을 위해 알아야 할 핵심 정보를 조사하세요.                                                       │
│  제주의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '제주도 지역별 맛집 추천 2024 2025'}                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 일정 기획자                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  안녕하세요! 10년 차 여행 플래너입니다. 제주도의 매력을 알차게 느끼실 수 있도록, 동선을 최적화하여 2박 3일      │
│  여행 일정을 기획해 드립니다. 이 일정은 **'제주 동부와 서귀포를 아우르는 핵심 코스'**로 구성하였습니다.         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### [1일 차: 제주 도착 및 동부의 절경 탐방]                                                                    │
│  **테마: 제주의 푸른 바다와 미디어 아트**                                                                       │
│                                                                                                                 │
│  *   **오전: 제주 도착 및 든든한 시작**                                                                         │
│      *   **일정:** 제주 공항 도착 후 렌터카 인수.                                                               │
│      *   **식사:** **우진해장국**에서 고사리 육개장으로 아침 식사. (웨이팅이 길 수 있으니 도착 직후             │
│  '캐치테이블' 앱으로 대기 현황을 확인하세요.)                                                                   │
│  *   **오후: 몰입형 전시와 산책**                                                                               │
│      *   **일정:** **아르떼뮤지엄 제주** 방문. 날씨에 상관없이 화려한 미디어 아트를 즐길 수 있습니다. 이후      │
│  **스누피가든**으로 이동하여 자연 속에서 여유로운 산책과 인생 사진을 남겨보세요.                                │
│  *   **저녁: 동부로 이동 및 휴식**                                                                              │
│      *   **일정:** 성산 근처 숙소로 이동.                                                                       │
│      *   **식사:** **고집돌우럭**에서 제주식 우럭 조림으로 푸짐한 저녁 식사.                                    │
│                                                                                                                 │
│  ### [2일 차: 제주의 자연과 액티비티]                                                                           │
│  **테마: 성산의 일출과 서귀포의 맛**                                                                            │
│                                                                                                                 │
│  *   **오전: 제주의 상징과 함께**                                                                               │
│      *   **일정:** **성산일출봉** 등반 혹은 주변 해안 산책. (일찍 방문하면 인파를 피해 여유롭게 절경을 감상할   │
│  수 있습니다.)                                                                                                  │
│      *   **식사:** **맛나식당**에서 갈치조림. (예약 필수이므로 전날 혹은 당일 아침 일찍 예약 가능 여부를        │
│  확인하세요.)                                                                                                   │
│  *   **오후: 액티비티와 전복 요리**                                                                             │
│      *   **일정:** **9.81 파크**에서 무동력 레이싱을 즐기며 스트레스를 해소하세요. 이후 서귀포 시내로           │
│  이동합니다.                                                                                                    │
│      *   **식사:** **오가네전복설렁탕**에서 전복 요리로 건강한 한 끼를 즐기세요.                                │
│  *   **저녁: 서귀포의 밤**                                                                                      │
│      *   **일정:** 서귀포 올레시장 구경 및 야식 구매. 숙소로 돌아와 제주 로컬 맥주와 함께 하루를 마무리합니다.  │
│                                                                                                                 │
│  ### [3일 차: 오름 투어와 여유로운 마무리]             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 위의 조사 결과를 참고하여 제주에서 3일 동안 머무는 여행 일정을 작성해 주세요.                            │
│  각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요.                       │
│  일정에는 방문지에 대한 간단한 설명이나 여행 팁도 포함해 주세요.                                                │
│  Agent: 여행 일정 기획자                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 97c0f2c3-505c-44c8-8c46-45692c5e50eb                                                                       │
│  Final Output: 안녕하세요! 10년 차 여행 플래너입니다. 제주도의 매력을 알차게 느끼실 수 있도록, 동선을           │
│  최적화하여 2박 3일 여행 일정을 기획해 드립니다. 이 일정은 **'제주 동부와 서귀포를 아우르는 핵심 코스'**로      │
│  구성하였습니다.                                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### [1일 차: 제주 도착 및 동부의 절경 탐방]                                                                    │
│  **테마: 제주의 푸른 바다와 미디어 아트**                                                                       │
│                                                                                                                 │
│  *   **오전: 제주 도착 및 든든한 시작**                                                                         │
│      *   **일정:** 제주 공항 도착 후 렌터카 인수.                                                               │
│      *   **식사:** **우진해장국**에서 고사리 육개장으로 아침 식사. (웨이팅이 길 수 있으니 도착 직후             │
│  '캐치테이블' 앱으로 대기 현황을 확인하세요.)                                                                   │
│  *   **오후: 몰입형 전시와 산책**                                                                               │
│      *   **일정:** **아르떼뮤지엄 제주** 방문. 날씨에 상관없이 화려한 미디어 아트를 즐길 수 있습니다. 이후      │
│  **스누피가든**으로 이동하여 자연 속에서 여유로운 산책과 인생 사진을 남겨보세요.                                │
│  *   **저녁: 동부로 이동 및 휴식**                                                                              │
│      *   **일정:** 성산 근처 숙소로 이동.                                                                       │
│      *   **식사:** **고집돌우럭**에서 제주식 우럭 조림으로 푸짐한 저녁 식사.                                    │
│                                                                                                                 │
│  ### [2일 차: 제주의 자연과 액티비티]                                                                           │
│  **테마: 성산의 일출과 서귀포의 맛**                                                                            │
│                                                                                                                 │
│  *   **오전: 제주의 상징과 함께**                                                                               │
│      *   **일정:** **성산일출봉** 등반 혹은 주변 해안 산책. (일찍 방문하면 인파를 피해 여유롭게 절경을 감상할   │
│  수 있습니다.)                                                                                                  │
│      *   **식사:** **맛나식당**에서 갈치조림. (예약 필수이므로 전날 혹은 당일 아침 일찍 예약 가능 여부를        │
│  확인하세요.)                                                                                                   │
│  *   **오후: 액티비티와 전복 요리**                                                                             │
│      *   **일정:** **9.81 파크**에서 무동력 레이싱을 즐기며 스트레스를 해소하세요. 이후 서귀포 시내로           │
│  이동합니다.                                                                                                    │
│      *   **식사:** **오가네전복설렁탕**에서 전복 요리로 건강한 한 끼를 즐기세요.                                │
│  *   **저녁: 서귀포의 밤**                                                                                      │
│      *   **일정:** 서귀포 올레시장 구경 및 야식 구매. 숙소로 돌아와 제주 로컬 맥주와 함께 하루를 마무리합니다.  │
│                                  

=== [협업 에이전트] 생성된 제주 3일 일정 ===
안녕하세요! 10년 차 여행 플래너입니다. 제주도의 매력을 알차게 느끼실 수 있도록, 동선을 최적화하여 2박 3일 여행 일정을 기획해 드립니다. 이 일정은 **'제주 동부와 서귀포를 아우르는 핵심 코스'**로 구성하였습니다.

---

### [1일 차: 제주 도착 및 동부의 절경 탐방]
**테마: 제주의 푸른 바다와 미디어 아트**

*   **오전: 제주 도착 및 든든한 시작**
    *   **일정:** 제주 공항 도착 후 렌터카 인수.
    *   **식사:** **우진해장국**에서 고사리 육개장으로 아침 식사. (웨이팅이 길 수 있으니 도착 직후 '캐치테이블' 앱으로 대기 현황을 확인하세요.)
*   **오후: 몰입형 전시와 산책**
    *   **일정:** **아르떼뮤지엄 제주** 방문. 날씨에 상관없이 화려한 미디어 아트를 즐길 수 있습니다. 이후 **스누피가든**으로 이동하여 자연 속에서 여유로운 산책과 인생 사진을 남겨보세요.
*   **저녁: 동부로 이동 및 휴식**
    *   **일정:** 성산 근처 숙소로 이동.
    *   **식사:** **고집돌우럭**에서 제주식 우럭 조림으로 푸짐한 저녁 식사.

### [2일 차: 제주의 자연과 액티비티]
**테마: 성산의 일출과 서귀포의 맛**

*   **오전: 제주의 상징과 함께**
    *   **일정:** **성산일출봉** 등반 혹은 주변 해안 산책. (일찍 방문하면 인파를 피해 여유롭게 절경을 감상할 수 있습니다.)
    *   **식사:** **맛나식당**에서 갈치조림. (예약 필수이므로 전날 혹은 당일 아침 일찍 예약 가능 여부를 확인하세요.)
*   **오후: 액티비티와 전복 요리**
    *   **일정:** **9.81 파크**에서 무동력 레이싱을 즐기며 스트레스를 해소하세요. 이후 서귀포 시내로 이동합니다.
    *   **식사:** **오가네전복설렁탕**에서 전복 요리로 건강한 한 끼를 즐기세요.
* 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
from torch._dynamo import aot_compile_types
from crewai_tools import ScrapeWebsiteTool 
import ast
import operator
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from crewai.tools import tool
from crewai import Agent, Task, Crew, Process

# 도구 인스턴스 생성
scrape_tool = ScrapeWebsiteTool(website_url="https://www.tripadvisor.co.kr/")

class CalculatorInput(BaseModel):
    expression: str = Field(..., description="계산할 수학 식 (예: '2 * (3 + 4)')")
class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = "안전하게 수학 계산을 수행합니다."
    args_schema: Type[BaseModel] = CalculatorInput
    def _run(self, expression: str) -> str:
        # 1. 허용할 연산자 매핑 정의
        allowed_operators = {
            ast.Add: operator.add,      # +
            ast.Sub: operator.sub,      # -
            ast.Mult: operator.mul,     # *
            ast.Div: operator.truediv,  # /
            ast.Pow: operator.pow,      # ** (거듭제곱)
            ast.USub: operator.neg,     # 단항 음수 (예: -5)
            ast.UAdd: operator.pos,     # 단항 양수 (예: +5)
        }
        # 2. AST 노드를 순회하며 안전한 노드만 재귀적으로 평가하는 헬퍼 함수
        def eval_node(node):
            if isinstance(node, ast.Expression):
                return eval_node(node.body)
            # 숫자(정수 및 실수) 상수 노드 허용
            elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
                return node.value
            # 사칙연산 등 이항 연산자 처리
            elif isinstance(node, ast.BinOp):
                left = eval_node(node.left)
                right = eval_node(node.right)
                op_type = type(node.op)
                if op_type in allowed_operators:
                    return allowed_operators[op_type](left, right)  #  표준 파이썬 호출 방식
                raise TypeError(f"지원하지 않는 연산자입니다: {op_type.__name__}")
            # 음수(-) 등 단항 연산자 처리
            elif isinstance(node, ast.UnaryOp):
                operand = eval_node(node.operand)
                op_type = type(node.op)
                if op_type in allowed_operators:
                    return allowed_operators[op_type](operand)
                raise TypeError(f"지원하지 않는 단항 연산자입니다: {op_type.__name__}")
            else:
                raise TypeError(f"허용되지 않는 구문(보안 경고): {type(node).__name__}")
        try:
            # 문자열 수식을 안전하게 AST 객체로 파싱 (코드가 실행되지 않음)
            tree = ast.parse(expression, mode="eval")
            result = eval_node(tree)
            return str(result)
        except Exception as e:
            return f"계산 오류: {e}"


@tool('calculator')
def calculator(expression: str) -> str:
    '''수학 계산을 수행합니다.'''
    try:
        result = eval(expression, {'__builtins__': {}})
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

calculator_tool = CalculatorTool()

travel_agent = Agent(
    role="여행 전문가",
    goal="최적의 여행 일정과 예산 계획 제공",
    backstory="다년간의 여행 플래너 경험 보유",
    tools=[search_tool, scrape_tool, calculator_tool] ,
    llm="gpt-5.4-mini",
    verbose=True
)

travel_task = Task(
    description="{place} {days}일 여행 일정(문화, 미식 포함), 예산은 {budget}달러 (항공 {airfare}달러, 숙박 하루 {accommodation_per_day}달러).",
    expected_output="한국어로 작성된 {days}일간 {place} 여행에 대한 상세한 일정과 예산 계산 결과",
    agent=travel_agent
)

crew = Crew(
    agents=[travel_agent],
    tasks=[travel_task],
    process=Process.sequential, 
    verbose=True
)
result = await crew.kickoff_async(inputs={'place': '삿포로', 'days':'7', 'budget':'1000', 'airfare':'100', "accommodation_per_day":'10'})
print('[여행 시나리오 응답]\n', result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9354bdd0-8f24-4694-a183-a546c09f69ec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 삿포로 7일 여행 일정(문화, 미식 포함), 예산은 1000달러 (항공 100달러, 숙박 하루 10달러).                 │
│  ID: ff4f17d1-2497-4dfd-99eb-da319c76be09                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│  Task: 삿포로 7일 여행 일정(문화, 미식 포함), 예산은 1000달러 (항공 100달러, 숙박 하루 10달러).                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculator executed with result: 830...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: calculator                                                                                               │
│  Output: 830                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  아래는 **삿포로 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**입니다.                                  │
│  전제 조건은 다음으로 반영했습니다.                                                                             │
│                                                                                                                 │
│  - **총 예산:** 1,000달러                                                                                       │
│  - **항공:** 100달러                                                                                            │
│  - **숙박:** 하루 10달러 × 7박 = 70달러                                                                         │
│                                                                                                                 │
│  즉, 항공과 숙박을 제외한 **남은 여행 예산은 830달러**입니다.                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1) 예산 계산 결과                                                                                            │
│                                                                                                                 │
│  ## 총예산                                                                                                      │
│  - 1,000달러                                                                                                    │
│                                                                                                                 │
│  ## 고정비                                                                                                      │
│  - 항공: 100달러                                                                                                │
│  - 숙박: 10달러 × 7일 = 70달러                                                                                  │
│                                                                                                                 │
│  ## 남는 예산                                                                                                   │
│  - 1,000 - 100 - 70 = **830달러**                                                                               │
│                                                                                                                 │
│  ### 1일 평균 사용 가능 금액                                                                                    │
│  - 830달러 ÷ 7일 = **약 118.6달러/일**                                                                          │
│                                                                                                                 │
│  이 예산이면 삿포로에서 **대중교통 + 지역 미식 + 주요 문화 관광**을 충분히 즐길 수 있습니다.                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 2) 7일 상세 일정                                                                                             │

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 삿포로 7일 여행 일정(문화, 미식 포함), 예산은 1000달러 (항공 100달러, 숙박 하루 10달러).                 │
│  Agent: 여행 전문가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[여행 시나리오 응답]
 아래는 **삿포로 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**입니다.  
전제 조건은 다음으로 반영했습니다.

- **총 예산:** 1,000달러
- **항공:** 100달러
- **숙박:** 하루 10달러 × 7박 = 70달러

즉, 항공과 숙박을 제외한 **남은 여행 예산은 830달러**입니다.

---

# 1) 예산 계산 결과

## 총예산
- 1,000달러

## 고정비
- 항공: 100달러
- 숙박: 10달러 × 7일 = 70달러

## 남는 예산
- 1,000 - 100 - 70 = **830달러**

### 1일 평균 사용 가능 금액
- 830달러 ÷ 7일 = **약 118.6달러/일**

이 예산이면 삿포로에서 **대중교통 + 지역 미식 + 주요 문화 관광**을 충분히 즐길 수 있습니다.

---

# 2) 7일 상세 일정

## Day 1. 삿포로 도착 + 시내 적응 + 대표 미식
### 오전/이동
- 삿포로 도착 후 숙소 체크인
- 첫날은 무리하지 말고 시내 중심부 위주로 동선 정리

### 오후
- **오도리 공원** 산책
- **삿포로 TV타워** 외관/전망 감상
- **삿포로 시계탑** 방문  
  - 삿포로의 상징적인 역사 건축물
  - 가볍게 도시 분위기를 익히기 좋음

### 저녁
- **스스키노(すすきの)**에서 첫 저녁
- 추천 미식:
  - **징기스칸(양고기 구이)**
  - **미소라멘**
  - **홋카이도 생맥주**
- 밤에는 스스키노 거리 야경 산책

### 예상 지출
- 교통: 5~10달러
- 식비: 25~40달러
- 입장료: 0~5달러

---

## Day 2. 역사와 문화: 홋카이도 청사 + 구도심 탐방
### 오전
- **홋카이도청 구본청사(붉은 벽돌 청사)**  
  - 삿포로의 대표적인 근대 건축
  - 사진 찍기 좋은 장소
- **홋카이도대학 캠퍼스 산책**
  - 계절별 풍경이 아름다움
  - 조용하고 학구적인 분위기

### 오후
- **삿포로 박물관*

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9354bdd0-8f24-4694-a183-a546c09f69ec                                                                       │
│  Final Output: 아래는 **삿포로 7일 여행 일정(문화 + 미식 중심)**과 **예산 계산 결과**입니다.                    │
│  전제 조건은 다음으로 반영했습니다.                                                                             │
│                                                                                                                 │
│  - **총 예산:** 1,000달러                                                                                       │
│  - **항공:** 100달러                                                                                            │
│  - **숙박:** 하루 10달러 × 7박 = 70달러                                                                         │
│                                                                                                                 │
│  즉, 항공과 숙박을 제외한 **남은 여행 예산은 830달러**입니다.                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 1) 예산 계산 결과                                                                                            │
│                                                                                                                 │
│  ## 총예산                                                                                                      │
│  - 1,000달러                                                                                                    │
│                                                                                                                 │
│  ## 고정비                                                                                                      │
│  - 항공: 100달러                                                                                                │
│  - 숙박: 10달러 × 7일 = 70달러                                                                                  │
│                                                                                                                 │
│  ## 남는 예산                                                                                                   │
│  - 1,000 - 100 - 70 = **830달러**                                                                               │
│                                                                                                                 │
│  ### 1일 평균 사용 가능 금액                                                                                    │
│  - 830달러 ÷ 7일 = **약 118.6달러/일**                                                                          │
│                                                                                                                 │
│  이 예산이면 삿포로에서 **대중교통 + 지역 미식 + 주요 문화 관광**을 충분히 즐길 수 있습니다.                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # 2) 7일 상세 일정                                                                                        

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯